# Foot detector — SynFoot pretrain (Colab GPU)

Run top to bottom on a Colab GPU runtime (T4/A100).
Repo must include `training/foot-detector/datasets/synfoot/` (43,742 train / 6,258 val)
— either `git clone` + Drive zip upload, or upload the zip built by `package_synfoot_zip.py`.
Source of truth for flags: `training/foot-detector/TRAIN_COLAB.md`.

In [ ]:
!nvidia-smi -L
!python -c "import torch; print('cuda:', torch.cuda.is_available())"
!pip install -r training/foot-detector/requirements.txt

In [ ]:
# Verify datasets made it onto the VM (counts must match the validated build)
!python -c "import os; b='training/foot-detector/datasets/synfoot'; print({s: {k: len(os.listdir(os.path.join(b,k,s))) for k in ['images','labels']} for s in ['train','val']})"
# expect train 43742 / val 6258

In [ ]:
# Ultralytics resolves a relative `path:` against its own datasets folder,
# not against this yaml — rewrite it to the absolute VM path or train fails
# with a doubled `datasets/datasets/...` lookup.
p = 'training/foot-detector/data-synfoot.yaml'
t = open(p).read().replace('path: datasets/synfoot', 'path: /content/training/foot-detector/datasets/synfoot')
open(p, 'w').write(t)
print(open(p).read())

In [ ]:
# Pretrain on SynFoot (~30-60 min on A100/T4, 100 epochs @ 640)
!yolo pose train \
  data=training/foot-detector/data-synfoot.yaml \
  model=yolo11n-pose.pt \
  epochs=100 imgsz=640 batch=16 \
  hsv_h=0.015 hsv_s=0.4 hsv_v=0.4 \
  degrees=15 translate=0.1 scale=0.3 \
  mosaic=0.5 mixup=0.1 copy_paste=0.1 \
  project=training/foot-detector/runs name=synfoot-pretrain

In [ ]:
# Validate (aggregate mAP — hard-slice scoring on dim/blur comes after fine-tune)
!yolo pose val data=training/foot-detector/data-synfoot.yaml model=training/foot-detector/runs/synfoot-pretrain/weights/best.pt

In [ ]:
# Export for the browser (opset 12 = onnxruntime-web coverage)
!yolo export model=training/foot-detector/runs/synfoot-pretrain/weights/best.pt format=onnx opset=12 dynamic=False
!ls -lh training/foot-detector/runs/synfoot-pretrain/weights/